# 04 — SageMaker Pipeline (DAG): Preprocess → Train → Evaluate → Register

This notebook builds and runs a **SageMaker Pipeline** that:

1. Preprocesses curated CSV into supervised features and train/val/test splits
2. Trains a scikit-learn model in a SageMaker Training Job
3. Evaluates the model and writes `evaluation.json`
4. **Conditionally registers** the model to **SageMaker Model Registry**
   - Pass = registers model
   - Fail = pipeline fails (great for demoing a failed DAG)

This satisfies:
- **Pipeline/DAG** requirement
- **Model Registry** requirement


In [1]:
%pip install -q -r ../docker/requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

import time
import boto3
import sagemaker

from src.pipelines.buoycast_pipeline import get_pipeline

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
# Load from previous notebook
%store -r bucket
%store -r region
%store -r S3_PREFIX_PARQUET
%store -r S3_PREFIX_CSV
%store -r BUOY_IDS
%store -r manifest_s3_uri

print("Bucket:", bucket)
print("Region:", region)
print("S3_PREFIX_PARQUET:", S3_PREFIX_PARQUET)
print("S3_PREFIX_CSV:", S3_PREFIX_CSV)
print("BUOY_IDS:", BUOY_IDS)
print("Manifest (S3):", manifest_s3_uri)

Bucket: sagemaker-us-east-1-318401170150
Region: us-east-1
S3_PREFIX_PARQUET: curated/ndbc_parquet
S3_PREFIX_CSV: curated/ndbc_csv
BUOY_IDS: ['46086', '46042', '46011']
Manifest (S3): s3://sagemaker-us-east-1-318401170150/manifests/buoy=all/curated_manifest.csv


In [4]:
import time
import sagemaker

# SageMaker context
sess = sagemaker.Session()
role = sagemaker.get_execution_role()

#curated_s3_prefix = f"s3://{bucket}/{S3_PREFIX_PARQUET}/"   # Parquet for pipeline
curated_s3_prefix = f"s3://{bucket}/{S3_PREFIX_CSV}/"

artifacts_prefix = f"s3://{bucket}/buoycast/artifacts"

PIPELINE_NAME = "buoycast-train-eval"   # rename (no register in pipeline)
MODEL_PACKAGE_GROUP = "buoycast-wave-models"  # still used later when we register manually

# RunId must be a REAL string (not a pipeline variable)
RUN_ID = time.strftime("%Y%m%d-%H%M%S")

print("Role:", role)
print("Curated:", curated_s3_prefix)
print("Artifacts:", artifacts_prefix)
print("RUN_ID:", RUN_ID)
print("ModelPackageGroup:", MODEL_PACKAGE_GROUP)


Role: arn:aws:iam::318401170150:role/LabRole
Curated: s3://sagemaker-us-east-1-318401170150/curated/ndbc_csv/
Artifacts: s3://sagemaker-us-east-1-318401170150/buoycast/artifacts
RUN_ID: 20260223-010544
ModelPackageGroup: buoycast-wave-models


In [5]:
# Create / update pipeline definition
pipeline = get_pipeline(
    region=region,
    role=role,
    default_bucket=bucket,
    pipeline_name=PIPELINE_NAME,
    base_job_prefix="buoycast",
)

pipeline.upsert(role_arn=role)
print("Upserted pipeline:", PIPELINE_NAME)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/steps.py:485: UserWarning: Profiling is enabled on the provided estimator. The default profiler rule includes a timestamp which will change each time the pipeline is upserted, causing cache misses. If profiling is not needed, set disable_profiler to True on the estimator.
  warnings.warn(msg)
INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


Upserted pipeline: buoycast-train-eval


In [6]:
execution = pipeline.start(
    parameters={
        "CuratedS3Prefix": curated_s3_prefix,
        "ArtifactsS3Prefix": artifacts_prefix,
        "RunId": RUN_ID,
        "RmseHsThreshold": 0.5,
        "ProcessingInstanceType": "ml.m5.xlarge",
        "TrainingInstanceType": "ml.m5.xlarge",
    }
)

print("PipelineExecutionArn:", execution.arn)


PipelineExecutionArn: arn:aws:sagemaker:us-east-1:318401170150:pipeline/buoycast-train-eval/execution/cclipxfs0utb


In [7]:
import time
import boto3

sm = boto3.client("sagemaker")

print("Monitoring pipeline execution...")

while True:
    desc = execution.describe()
    status = desc["PipelineExecutionStatus"]
    print("Pipeline status:", status)

    steps = sm.list_pipeline_execution_steps(
        PipelineExecutionArn=execution.arn
    )["PipelineExecutionSteps"]

    for s in steps:
        print("  -", s["StepName"], ":", s["StepStatus"])

    print("-----")

    if status in ["Succeeded", "Failed", "Stopped"]:
        break

    time.sleep(20)

print("\nFinal status:", status)


Monitoring pipeline execution...
Pipeline status: Executing
-----
Pipeline status: Executing
  - Preprocess : Executing
-----
Pipeline status: Executing
  - Preprocess : Executing
-----
Pipeline status: Executing
  - Preprocess : Executing
-----
Pipeline status: Executing
  - Preprocess : Executing
-----
Pipeline status: Executing
  - Preprocess : Executing
-----
Pipeline status: Executing
  - Preprocess : Executing
-----
Pipeline status: Executing
  - Preprocess : Executing
-----
Pipeline status: Executing
  - Train : Executing
  - Preprocess : Succeeded
-----
Pipeline status: Executing
  - Train : Executing
  - Preprocess : Succeeded
-----
Pipeline status: Executing
  - Train : Executing
  - Preprocess : Succeeded
-----
Pipeline status: Executing
  - Train : Executing
  - Preprocess : Succeeded
-----
Pipeline status: Executing
  - Train : Executing
  - Preprocess : Succeeded
-----
Pipeline status: Executing
  - Train : Executing
  - Preprocess : Succeeded
-----
Pipeline status: Execu

In [8]:
pipeline_execution_arn = execution.arn

%store PIPELINE_NAME
%store MODEL_PACKAGE_GROUP
%store RUN_ID
%store pipeline_execution_arn


Stored 'PIPELINE_NAME' (str)
Stored 'MODEL_PACKAGE_GROUP' (str)
Stored 'RUN_ID' (str)
Stored 'pipeline_execution_arn' (str)


In [12]:
import boto3, sagemaker
from sagemaker.model_metrics import ModelMetrics, MetricsSource
from sagemaker.sklearn.model import SKLearnModel

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
sm = boto3.client("sagemaker")

# 1) find the training job from this pipeline execution
steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=pipeline_execution_arn)["PipelineExecutionSteps"]
train_step = next(s for s in steps if "TrainingJob" in s.get("Metadata", {}))
training_job_name = train_step["Metadata"]["TrainingJob"]["Arn"].split("/")[-1]
tj = sm.describe_training_job(TrainingJobName=training_job_name)

model_data = tj["ModelArtifacts"]["S3ModelArtifacts"]

# 2) point to evaluation.json written by your eval step
evaluation_s3_uri = f"{artifacts_prefix}/{RUN_ID}/reports/evaluation/evaluation.json"

print("TrainingJob:", training_job_name)
print("ModelData:", model_data)
print("Eval:", evaluation_s3_uri)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(s3_uri=evaluation_s3_uri, content_type="application/json")
)

# IMPORTANT: use SKLearnModel so we can include entry_point/source_dir
sk_model = SKLearnModel(
    model_data=model_data,
    role=role,
    entry_point="inference.py",
    source_dir="../src/inference",
    framework_version="1.2-1",  # if this mismatches, set to your training framework version
    py_version="py3",
    sagemaker_session=sess,
)

mp = sk_model.register(
    model_package_group_name=MODEL_PACKAGE_GROUP,
    model_metrics=model_metrics,
    approval_status="PendingManualApproval",
    content_types=["text/csv", "application/json"],
    response_types=["application/json"],
    inference_instances=["ml.m5.large", "ml.m5.xlarge"],
    transform_instances=["ml.m5.large"],
)

print("Registered model package ARN:", mp.model_package_arn)


TrainingJob: pipelines-cclipxfs0utb-Train-bY0s9ows1Q
ModelData: s3://sagemaker-us-east-1-318401170150/pipelines-cclipxfs0utb-Train-bY0s9ows1Q/output/model.tar.gz
Eval: s3://sagemaker-us-east-1-318401170150/buoycast/artifacts/20260223-010544/reports/evaluation/evaluation.json
Registered model package ARN: arn:aws:sagemaker:us-east-1:318401170150:model-package/buoycast-wave-models/2


In [13]:
# Save for next notebooks
MODEL_PACKAGE_ARN = mp.model_package_arn
%store MODEL_PACKAGE_ARN

Stored 'MODEL_PACKAGE_ARN' (str)
